# Writing CatBoost from Scratch: Newton Steps and L2 Scoring

This notebook looks under the hood of gradient boosting. We build a custom Python engine from scratch for CatBoost CPU/Plain mode: numerical features, unit object weights, symmetric trees, and one Newton leaf-estimation step. Results were checked with CatBoost 1.2.8; agreement on these examples does not imply support for every library mode.

**We will cover:**
* Compute **L2-score and Cosine** to find optimal splits.
* Computing **Newton step (Newton leaf estimation)** to update probabilities.

In [ ]:
# Dependencies: pip install -r requirements.txt
# Checked with: CatBoost 1.2.8, NumPy 1.26.4, pandas 2.2.3.

import os
import io
import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier, Pool

warnings.filterwarnings('ignore')

# For Windows users with Graphviz rendering issues, uncomment the line below and set your own path:
# os.environ["PATH"] += os.pathsep + r'C:\Users\YourName\Graphviz\bin'

## 1. Custom Boosting Engine

The main functions are: leaf-mask search, best-split calculation by maximizing `L2-score` or `Cosine` and probability updates using `Newton leaf estimation`.

We also wrote a general-purpose function `parse_borders` to extract quantization borders directly from a CatBoost file. To avoid duplicating code, we placed the logic in `fit_custom_catboost`.

In [ ]:
def fit_custom_catboost(df, features, target, borders_dict, iterations, depth, learning_rate, l2_reg, score_function='L2', verbose=True, drawing_mode=False, score_calcer=None):
    """
    Main function for training a custom gradient-boosting model.

    The algorithm builds decision trees sequentially. Each new tree
    tries to correct errors made by all previous trees.

    Arguments:
        df (pandas.DataFrame): Training data (table).
        features (list): List of feature (column) names used to build rules.
        target (str): Target variable (column to predict).
        borders_dict (dict): Dictionary of borders (thresholds) for splitting feature values.
        iterations (int): Number of trees to build (algorithm epochs / steps).
        depth (int): Maximum depth of each tree (number of levels).
        learning_rate (float): Learning rate. Smaller steps train the model more cautiously.
        l2_reg (float): L2 regularization coefficient (a penalty that helps prevent overfitting).
        score_function (str): Split-quality scoring function ('L2' or 'Cosine').
        verbose (bool): Whether to print training progress information.
        drawing_mode (bool): If True, return both predictions and all trees.
        score_calcer (callable | None): Separate criterion for the educational experiment.
            calculate_score is used by default; the global function is not replaced.
    """
    # Copy the source data to avoid accidental modification
    df = df.copy()

    # Keep the specified starting probabilities (baseline) for the Hessian experiment (used once in the article)
    # If predictions are not available, start every row at 50% (0.5) (the model knows nothing yet)
    if 'pred_prob' not in df.columns:
        df['pred_prob'] = 0.5

    # Store the structure (rules) of all built trees
    all_trees = []

    # Main training loop: build trees one by one
    for i in range(iterations):
        if verbose:
            print(f"\n{'='*60}\nTree {i+1}\n{'='*60}")

        # The anti-gradient (in the gradient column) is the direction and magnitude of the correction: how far the true answer is from our prediction
        df['gradient'] = df[target] - df['pred_prob']

        # The Hessian is the second derivative of the loss with respect to the logit.
        # For probability prediction it is computed as: probability * (1 - probability)
        df['hessian'] = df['pred_prob'] * (1 - df['pred_prob'])

        # List storing the current tree rules (splits)
        tree = []

        # Build the tree level by level until maximum depth (depth)
        for d in range(depth):
            # 🌟 PASSING score_function HERE:
            # Find the best rule (split) to divide the data into two parts
            best_split = find_best_split(df, tree, features, borders_dict, l2_reg, score_function, score_calcer)

            # If no split improves quality, stop growing this tree
            if best_split is None:
                if verbose:
                    print(f"  [✂️ CROPPED] Stopped at depth {d} (no splits improving {score_function}-score)")
                break

            # Add the best split to the tree structure
            tree.append(best_split)

        # Print the built tree structure
        if verbose:
            print("  Structure:")
            if not tree:
                print("    [Empty Tree]")
            for lvl, (f, v) in enumerate(tree):
                # Print each rule: level, feature, and threshold
                print(f"    Level {lvl+1}: {f} <= {v}")
            print("\n  Leaves Applied:")

        # Update dataset predictions using the newly built tree
        df['pred_prob'] = update_predictions(df, tree, learning_rate, l2_reg, verbose)
        all_trees.append(tree)

    # Return results according to drawing_mode
    if drawing_mode:
        return df['pred_prob'], all_trees
    else:
        return df['pred_prob']

def calculate_score(df, masks_list, score_function, l2_reg):
    """
    Compute the score of a proposed split into leaves.
    A larger score means the split better separates the classes.

    Arguments:
        df (pandas.DataFrame): Data table containing computed errors ('gradient').
        masks_list (list): List of filters (masks). Each shows which rows reached a leaf.
        score_function (str): Scoring function name ('L2' or 'Cosine').
        l2_reg (float): L2 regularization coefficient (a penalty that helps prevent overfitting).
    """
    # 🌟 INTERNAL FUNCTION FOR TREE SCORE
    total_num = 0     # Numerator (matches the final L2 result)
    total_den_sq = 0  # Denominator under the square root (used only for Cosine)
    tmp_debug = []    # Intermediate debug list

    # Check each leaf (mask) created by the split
    for mask in masks_list:
        w = mask.sum()  # Number of objects (weight) in this leaf. Sum of True values.

        # If the leaf contains any data
        if w > 0:
            # Sum gradients (errors) for all rows in this leaf
            sum_grad = df.loc[mask, 'gradient'].sum()

            # a_leaf - potential (candidate) leaf value produced by the tree.
            # Formula: sum of gradients / (number of objects + L2 regularization).
            # l2_reg is passed as a function argument.
            a_leaf = sum_grad / (w + l2_reg)

            # Numerator: sum(w_i * a_i * g_i). Shows this leaf contribution.
            total_num += a_leaf * sum_grad

            # Denominator: sum(w_i * a_i^2). Accumulated to compute cosine similarity.
            total_den_sq += w * (a_leaf ** 2)
            tmp_debug.append([w, a_leaf, w * (a_leaf ** 2)])

    # Return the final score for the selected method
    if score_function == 'Cosine':
        # Normalize by the denominator square root (omit constant sum(g^2))
        return total_num / np.sqrt(total_den_sq) if total_den_sq > 0 else 0
    else:  # L2
        return total_num

def find_best_split(df, tree, features, borders_dict, l2_reg, score_function='L2', score_calcer=None):
    """
    Find the best feature and border for a new data split
    with support for L2 and Cosine score functions.

    The function tries all available options and selects the one that gives
    the largest improvement in prediction quality.
    """
    score_calcer = calculate_score if score_calcer is None else score_calcer

    # Get the current row-to-leaf assignment before the new split
    leaf_masks_dict = get_leaves_masks(df, tree)

    # 1. Compute the current tree score (BEFORE the new split), our comparison baseline
    current_score = score_calcer(df, leaf_masks_dict.values(), score_function, l2_reg)

    # Minimum threshold to beat. Add a tiny value (1e-9),
    # This is a stopping rule for the educational engine; it does not cover all CatBoost stopping cases.
    best_score = current_score + 1e-9
    best_split = None

    # 2. Try all features in order
    for feat_idx, feature in enumerate(features):
        # Skip features without prepared borders
        if feat_idx not in borders_dict:
            continue

        borders = borders_dict[feat_idx]

        # Try every possible threshold for the current feature
        for border in borders:
            # Create the left-branch mask: values below the threshold or empty (NaN) go left
            is_left = (df[feature] <= border) | df[feature].isna()

            # Skip options where all data go to one side (no split)
            if is_left.sum() == 0 or (~is_left).sum() == 0:
                continue

            # Collect new leaf masks after the hypothetical split
            new_masks = []
            for mask in leaf_masks_dict.values():
                # Split each old leaf into two new leaves: left and right
                new_masks.append(mask & is_left)
                new_masks.append(mask & (~is_left))

            # Compute the candidate tree score with this split
            total_score = score_calcer(df, new_masks, score_function, l2_reg)

            # If this option is better than all previous ones, keep it as the best
            if total_score > best_score:
                best_score = total_score
                best_split = (feature, border)

    return best_split

def parse_borders(filepath):
    """
    Parse a CatBoost quantization-border text file into a dictionary.

    Quantization avoids trying every unique feature value
    (for example, every possible salary) by dividing them into a small number of threshold bins.
    This greatly speeds up training.
    """
    borders_dict = {}

    with open(filepath, 'r') as f:
        # Read the file line by line
        for line in f:
            # Split the line on the tab character
            parts = line.strip().split('\t')

            # If the line contains both a feature index and a threshold
            if len(parts) >= 2:
                feat_idx = int(parts[0])      # Feature index
                border_val = float(parts[1])  # Threshold value

                # If this feature is not in the dictionary, create an empty list
                if feat_idx not in borders_dict:
                    borders_dict[feat_idx] = []

                # Append the threshold to that feature list
                borders_dict[feat_idx].append(border_val)

    return borders_dict

def get_leaves_masks(df, tree):
    """
    Return boolean row masks (True/False arrays) for each tree leaf.

    Pass source data through all built-tree conditions
    and determine which rows reached each final leaf.
    """
    # If the tree is empty (no rules), all data are in one large root leaf
    if not tree:
        return {"Root (All Data)": pd.Series(True, index=df.index)}

    # Start with one branch without rules containing every row (the mask is all True)
    list_of_branches = [([], pd.Series(True, index=df.index))]

    # Walk through each tree split rule (level) from top to bottom
    for column_to_check, split_value in tree:
        new_cut_pieces = []

        # Apply the new rule to every existing branch
        for current_rules, current_mask in list_of_branches:

            # Left-branch logic: the condition is true (value is below or equal to the threshold)
            is_left_side = (df[column_to_check] <= split_value) | df[column_to_check].isna()
            left_mask = current_mask & is_left_side
            new_cut_pieces.append((current_rules + [f"{column_to_check}<={split_value:.3f}"], left_mask))

            # Right-branch logic: the condition is false (value is strictly above the threshold)
            is_right_side = ~is_left_side
            right_mask = current_mask & is_right_side
            new_cut_pieces.append((current_rules + [f"{column_to_check}>{split_value:.3f}"], right_mask))

        # Replace old branches with the newly split branches for the next step
        list_of_branches = new_cut_pieces

    # Join rules with " & " (AND) to create a readable leaf name
    return {" & ".join(rules): mask for rules, mask in list_of_branches}


def update_predictions(df, tree, lr, l2_reg, verbose=True):
    """
    Compute leaf values with a Newton step and update model predictions.

    The algorithm computes the value produced by each tree leaf,
    and updates probabilities (log-odds) for all objects.
    """
    # Get the leaf assignment for our rows
    leaf_masks_dict = get_leaves_masks(df, tree)

    # Create an empty column for the newly computed probabilities
    new_probs = pd.Series(index=df.index, dtype=float)

    # Process each tree leaf in turn
    for leaf_name, mask in leaf_masks_dict.items():
        # If no objects reached this leaf, continue
        if not mask.any():
            continue

        # Sum gradients (errors) and Hessians ("confidence") for rows in this leaf
        sum_grad = df.loc[mask, 'gradient'].sum()
        sum_hess = df.loc[mask, 'hessian'].sum()

        # Newton-step formula for the leaf value.
        # Divide errors by "confidence" + L2 regularization (to avoid division by zero).
        # Multiply by lr (learning rate) to move gradually toward the target and avoid overfitting.
        leaf_value = (sum_grad / (sum_hess + l2_reg)) * lr

        if verbose:
            print(f"    Leaf Value: {leaf_value:>7.4f} | Rows: {mask.sum():>4} | Filters: [{leaf_name}]")

        # Take the previous predicted probabilities for objects in this leaf
        p = df.loc[mask, 'pred_prob']

        # Clipping: trim probabilities slightly so they are never exactly 0.0 or 1.0.
        # Otherwise the logarithm below would fail (logarithm of 0 is undefined).
        p_safe = np.clip(p, 1e-15, 1 - 1e-15)

        # Convert probabilities (0 to 1) to "log-odds" (from minus infinity to plus infinity).
        # Safely add the new tree output (leaf_value) to the log-odds.
        new_log_odds = np.log(p_safe / (1 - p_safe)) + leaf_value

        # Convert updated "log-odds" back to probabilities with the sigmoid function
        new_probs.loc[mask] = 1 / (1 + np.exp(-new_log_odds))

    # Return the updated prediction column for the full dataset
    return new_probs

## 2. Exploring the Math on Simple Data (Toy Dataset)

Generate a credit-risk dataset to see how the algorithm selects splits and computes leaf values. Add a plot for visualization.

In [ ]:
df_toy = pd.DataFrame({
    'person_income':       [100000, 80000, 60000, 120000, 50000, 150000,  30000, 45000, 80000, 25000, 70000,   90000, 110000,  35000, 40000],
    'loan_percent_income': [  0.10,  0.15,  0.05,   0.18,  0.12,   0.08,   0.40,  0.35,  0.50,  0.45,  0.55,    0.25,   0.28,   0.26,  0.29],
    'person_age':          [    25,    42,    55,     30,    48,     41,     22,    38,    60,    28,    33,      45,     24,     50,    35],
    'loan_status':         [     1,     1,     1,      1,     1,      1,      0,     0,     0,     0,     0,       1,      1,      0,     0]
})
target_toy = 'loan_status'
features_toy = ['loan_percent_income', 'person_income']

plt.figure(figsize=(10, 6))
status_1 = df_toy[df_toy['loan_status'] == 1]
status_0 = df_toy[df_toy['loan_status'] == 0]

plt.scatter(status_1['person_income'], status_1['loan_percent_income'],
            color='navy', marker='o', s=100, label='Repayment (1)')
plt.scatter(status_0['person_income'], status_0['loan_percent_income'],
            facecolors='none', edgecolors='navy', marker='o', s=100, linewidths=1.5, label='Default (0)')

plt.xlim(0, 180000)
plt.ylim(0, 0.7)
plt.xlabel('Income (person_income)')
plt.ylabel('Loan share (loan_percent_income)')
plt.title('Toy data reference plot')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

## 3. Train a standard CatBoost model

In [ ]:
# Set parameters
cb_params_toy = {
  'iterations': 2,
  'depth': 4,
  'learning_rate': 0.8,
  'random_seed': 1,
  'reg_lambda': 3.0,
  'bootstrap_type': 'No',
  'penalties_coefficient': 0,
  'langevin': False,
  'diffusion_temperature': 0,
  'random_strength': 0,
  'logging_level': 'Silent',
  'score_function': 'Cosine',
  'leaf_estimation_method': 'Newton'
}

In [ ]:
pool_toy = Pool(df_toy[features_toy], df_toy[target_toy])
pool_toy.quantize()
pool_toy.save_quantization_borders('border_toy.txt')
borders_dict_toy = parse_borders('border_toy.txt')


print("⏳ Training original CatBoost...")
log_buffer = io.StringIO()
model_toy = CatBoostClassifier(**cb_params_toy).fit(pool_toy,log_cout=log_buffer,logging_level='Debug')
l2_reg_toy = model_toy.get_all_params()['l2_leaf_reg']

In [ ]:
model_toy.plot_tree(
    tree_idx=0,
    pool=pool_toy
)

In [ ]:
print(log_buffer.getvalue())

In [ ]:
df_toy['Catboost_pred']=model_toy.predict_proba(pool_toy)[:, 1]

In [ ]:
df_toy

## 4. Train CatBoost with our code

In [ ]:
pool_toy = Pool(df_toy[features_toy], df_toy[target_toy])
pool_toy.quantize()
pool_toy.save_quantization_borders('border_toy.txt')
borders_dict_toy = parse_borders('border_toy.txt')

print("⏳ Training original CatBoost...")
model_toy = CatBoostClassifier(**cb_params_toy).fit(pool_toy)
l2_reg_toy = model_toy.get_all_params()['l2_leaf_reg']

print("\n🚀 Running manual implementation (Toy Dataset)...")
custom_pred_toy = fit_custom_catboost(
    df_toy, features_toy, target_toy, borders_dict_toy,
    cb_params_toy['iterations'], cb_params_toy['depth'],
    cb_params_toy['learning_rate'], l2_reg_toy,score_function=cb_params_toy['score_function']
)

cb_pred_toy = model_toy.predict_proba(pool_toy)[:, 1]
custom_values_toy = np.asarray(custom_pred_toy, dtype=float)
cb_values_toy = np.asarray(cb_pred_toy, dtype=float)
assert custom_values_toy.shape == cb_values_toy.shape
assert np.isfinite(custom_values_toy).all() and np.isfinite(cb_values_toy).all()
max_diff_toy = np.max(np.abs(cb_values_toy - custom_values_toy))

print("\n" + "="*80)
print(f"🎯 Maximum probability difference (Toy): {max_diff_toy:.12f}")
if max_diff_toy < 1e-7:
    print("✅ Predictions match within absolute tolerance 1e-7.")
else:
    print("❌ Differences found.")
print("="*80)

In [ ]:
df_toy['pred']=custom_pred_toy
df_toy

## 5. Production-style Titanic Test (NaN Trick)

Real datasets often contain missing values (`NaN`). By default, CatBoost uses the missing-value strategy `nan_mode='Min'`, which means that quantization and split search send all `NaN` to the far-left subtree (treating them as smaller than any other value).

To simulate this behavior with pure Pandas/NumPy math (Pandas/Numpy), we fill missing values with an extremely small number `-999`. For these features, −999 is below every observed value. This is explicit preprocessing, not a full reproduction of CatBoost NaN quantization. For true NaNs, split search and object routing must use the same convention; here they go left.

In [ ]:
print("⏳ Loading Titanic dataset...")
try:
    df_raw = sns.load_dataset('titanic')
except Exception:
    url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
    df_raw = pd.read_csv(url)
    df_raw.columns = [c.lower() for c in df_raw.columns]

features_tit = ['pclass', 'age', 'sibsp', 'parch', 'fare']
target_tit = 'survived'
df_titanic = df_raw[features_tit + [target_tit]].copy()

# HACK: Replace missing values with -999 to emulate nan_mode='Min'
df_titanic = df_titanic.fillna(-999)
print(f"Dataset size: {df_titanic.shape[0]} rows. All missing values replaced with -999.\n")

cb_params_tit = {
  'iterations': 10,
  'depth': 6,
  'learning_rate': 0.5,
  'random_seed': 42,
  'reg_lambda': 3.0,
  'bootstrap_type': 'No',
  'penalties_coefficient': 0,
  'langevin': False,
  'diffusion_temperature': 0,
  'random_strength': 0,
  'logging_level': 'Silent',
  'score_function': 'Cosine',
  'leaf_estimation_method': 'Newton'
}



print("⏳ Quantizing features...")
pool_tit = Pool(df_titanic[features_tit], df_titanic[target_tit])
pool_tit.quantize()
pool_tit.save_quantization_borders('titanic_borders.txt')
borders_dict_tit = parse_borders('titanic_borders.txt')

print("⏳ Training original CatBoost...")
model_tit = CatBoostClassifier(**cb_params_tit).fit(pool_tit)
l2_reg_tit = model_tit.get_all_params()['l2_leaf_reg']
cb_pred_tit = model_tit.predict_proba(pool_tit)[:, 1]

print("🚀 Running manual implementation (Custom Engine)...")
custom_pred_tit = fit_custom_catboost(
    df_titanic, features_tit, target_tit, borders_dict_tit,
    cb_params_tit['iterations'], cb_params_tit['depth'],
    cb_params_tit['learning_rate'], l2_reg_tit,cb_params_tit['score_function'],
    verbose=False  # Disable logs to keep output concise
)


custom_values_tit = np.asarray(custom_pred_tit, dtype=float)
cb_values_tit = np.asarray(cb_pred_tit, dtype=float)
assert custom_values_tit.shape == cb_values_tit.shape
assert np.isfinite(custom_values_tit).all() and np.isfinite(cb_values_tit).all()
max_diff_tit = np.max(np.abs(cb_values_tit - custom_values_tit))
print("\n" + "="*80)
print(f"🎯 Maximum probability difference (Titanic): {max_diff_tit:.12f}")
if max_diff_tit < 1e-7:
    print("✅ Titanic predictions match within absolute tolerance 1e-7.")
else:
    print("❌ Differences found.")
print("="*80)

# Geometric Experiment with a Modified Score Formula

In this separate experiment, replace the score candidate denominator: use the Hessian sum instead of the object count. This is an intentional change to the educational algorithm, not CatBoost standard L2/Cosine scoring. Compare the sensitivity of the unnormalized criterion and cosine normalization to candidate scale.

The experiment uses separate data and a separate function passed through `score_calcer`. The main `calculate_score` is unchanged. Overfitting on new data is not evaluated here.

In [ ]:
def calculate_experimental_score(df,masks_list,score_function,l2_reg):
    # 🌟 INTERNAL FUNCTION FOR TREE SCORE
        total_num = 0     # Numerator (matches L2-score)
        total_den_sq = 0  # Denominator under the square root for Cosine
        for mask in masks_list:
            w = mask.sum() # Number of objects (weight) in leaf
            if w > 0:
                sum_grad = df.loc[mask, 'gradient'].sum()
                # EXPERIMENT: intentionally replace object count with Hessian sum.
                sum_hess = df.loc[mask, 'hessian'].sum()
                # a_leaf - potential (candidate) value in the leaf
                a_leaf = sum_grad / (sum_hess + l2_reg)

                # Numerator: sum(w_i * a_i * g_i)
                total_num += a_leaf * sum_grad

                # Denominator: sum(w_i * a_i^2)
                total_den_sq += w * (a_leaf ** 2)
        if score_function == 'Cosine':
            # Normalize by the denominator square root (omit constant sum(g^2))
            return total_num / np.sqrt(total_den_sq) if total_den_sq > 0 else 0
        else: # L2
            return total_num


def get_simple_borders(df, features):
    borders_dict = {}
    for i, feat in enumerate(features):
        vals = sorted(df[feat].dropna().unique())
        # Take midpoints between all unique points for ideal splits
        borders = [(vals[j] + vals[j+1])/2 for j in range(len(vals)-1)]
        borders_dict[i] = borders
    return borders_dict

# ==========================================
# 2. DATA PREPARATION AND HESSIAN HACK
# ==========================================
np.random.seed(42)
N = 100

# Main clusters: perfectly separated by X1 (vertically)
X_0 = np.random.normal(loc=[-1.0, 0.0], scale=[0.3, 1.0], size=(N, 2))
X_1 = np.random.normal(loc=[1.0, 0.0], scale=[0.3, 1.0], size=(N, 2))
y_0 = np.zeros(N)
y_1 = np.ones(N)

# Trap outliers (Class 1, but placed high on X2)
X_out = np.array( [[-1.0, 5.0], [-1.1, 5.1], [-0.9, 4.9]])
y_out = np.array([1, 1, 1])

# Collect dataset
df_experiment = pd.DataFrame(np.vstack([X_0, X_1, X_out]), columns=['X1', 'X2'])
df_experiment['target'] = np.concatenate([y_0, y_1, y_out])

# HESSIAN HACK: Simulate overconfident error for outliers
df_experiment['pred_prob'] = 0.5
df_experiment.iloc[-3:, df_experiment.columns.get_loc('pred_prob')] = 0.001

features_experiment = ['X1', 'X2']
borders_dict_experiment = get_simple_borders(df_experiment, features_experiment)

#

# ==========================================
# 4. VISUALIZATION
# ==========================================
def plot_split(ax, title, tree, df):
    ax.scatter(df[df['target']==0]['X1'], df[df['target']==0]['X2'], c='#3498db', edgecolors='k', label='Class 0')
    ax.scatter(df[df['target']==1]['X1'][:-3], df[df['target']==1]['X2'][:-3], c='#e74c3c', edgecolors='k', label='Class 1')

    # Highlight outliers
    ax.scatter(df[df['target']==1]['X1'][-3:], df[df['target']==1]['X2'][-3:],
               c='#f1c40f', s=250, marker='*', edgecolors='red', linewidths=1.5, label='Outliers (Class 1)')

    # Draw split line
    if tree and len(tree[0]) > 0:
        feature, value = tree[0][0]
        if feature == 'X1':
            ax.axvline(x=value, color='green', linestyle='--', linewidth=3, label=f'Split: {feature} <= {value:.2f}')
        else:
            ax.axhline(y=value, color='black', linestyle='--', linewidth=3, label=f'Split: {feature} <= {value:.2f}')

    ax.set_title(title, fontsize=12, fontweight='bold', pad=15)
    ax.set_xlabel('Feature X1')
    ax.set_ylabel('Feature X2 (Noise)')
    ax.legend(loc='lower right')



In [ ]:
# Weaken the regularizer to let fractions explode!
l2_reg_experiment = 0.001

# ==========================================
# 3. TRAINING MODELS
# ==========================================
# Train L2
_, trees_l2 = fit_custom_catboost(
    df_experiment, features_experiment, 'target', borders_dict_experiment,
    iterations=1, depth=1, learning_rate=1.0, l2_reg=l2_reg_experiment, score_function='L2', verbose=True,drawing_mode=True,score_calcer=calculate_experimental_score
)

# Train Cosine
_, trees_cos = fit_custom_catboost(
    df_experiment, features_experiment, 'target', borders_dict_experiment,
    iterations=1, depth=1, learning_rate=1.0, l2_reg=l2_reg_experiment, score_function='Cosine', verbose=True,drawing_mode=True,score_calcer=calculate_experimental_score
)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_split(ax1, 'Unnormalized criterion (experiment)', trees_l2, df_experiment)
plot_split(ax2, 'Cosine criterion (experiment)', trees_cos, df_experiment)

plt.suptitle('Educational experiment: how candidate scale affects split selection', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

At lambda=0.001, the modified unnormalized criterion selected X2 and isolated three objects, while the cosine criterion selected X1. At lambda=0.1, both criteria select X1. This is a result of this modified-formula experiment, not evidence that CatBoost standard L2 scoring is deficient or that Cosine generalizes better.

In [ ]:
# At lambda=0.1 both criteria in this experiment select X1.
l2_reg_experiment = 0.1

# ==========================================
# 3. TRAINING MODELS
# ==========================================
# Train L2
_, trees_l2 = fit_custom_catboost(
    df_experiment, features_experiment, 'target', borders_dict_experiment,
    iterations=1, depth=1, learning_rate=1.0, l2_reg=l2_reg_experiment, score_function='L2', verbose=True,drawing_mode=True,score_calcer=calculate_experimental_score
)

# Train Cosine
_, trees_cos = fit_custom_catboost(
    df_experiment, features_experiment, 'target', borders_dict_experiment,
    iterations=1, depth=1, learning_rate=1.0, l2_reg=l2_reg_experiment, score_function='Cosine', verbose=True,drawing_mode=True,score_calcer=calculate_experimental_score
)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

plot_split(ax1, 'Unnormalized criterion (experiment)', trees_l2, df_experiment)
plot_split(ax2, 'Cosine criterion (experiment)', trees_cos, df_experiment)

plt.suptitle('Educational experiment: how candidate scale affects split selection', fontsize=16, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()